# FraudShield v3.0.0 — Real-Time Streaming, Graph & Monitoring

This notebook exercises the v3.0.0 real-time surface without requiring any
live infrastructure:

1. Pluggable broker abstraction (Kafka / Redpanda)
2. Stateful streaming features (`StatefulFeatureStore`)
3. Batch rolling features (same engine the preprocessor uses)
4. Fraud ring detection (Louvain communities over 2-hop subgraphs)
5. Hybrid risk engine (ML 0.6 / graph 0.3 / rules 0.1)
6. The FastAPI inference API with live Prometheus metrics

Cells that can use live infrastructure (Neo4j) degrade gracefully when it is
offline. Start the full stack with `docker compose -f infra/docker-compose.yml up -d`.

In [1]:
import os
from pathlib import Path

# Resolve the repository root so relative paths work from any launch directory.
candidate = Path.cwd()
PROJECT_ROOT = candidate if (candidate / "data" / "raw").exists() else next(
    (p for p in candidate.parents if (p / "data" / "raw").exists()), candidate
)
os.chdir(PROJECT_ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (10, 5)

import fraudshield

print("Project root :", PROJECT_ROOT)
print("Package      :", Path(fraudshield.__file__).resolve().parent)
print("numpy/pandas :", np.__version__, "/", pd.__version__)

Project root : /Users/mudit/Developer/FraudShield
Package      : /Users/mudit/Developer/FraudShield/src/fraudshield
numpy/pandas : 2.4.6 / 3.0.3


## 1. Pluggable Broker Abstraction

`FRAUDSHIELD_KAFKA_BROKER_TYPE` selects the factory. Both brokers speak the
Kafka wire protocol via `confluent-kafka`; Redpanda defaults to
`SASL_SSL` + `SCRAM-SHA-256`. SASL credentials are wired automatically when
`FRAUDSHIELD_KAFKA_SASL_USERNAME`/`..._PASSWORD` are set.

In [2]:
from fraudshield.config.settings import get_settings
from fraudshield.streaming.broker import get_broker_factory

settings = get_settings()
print("Configured broker type:", settings.kafka.broker_type)

kafka_producer = get_broker_factory("kafka").create_producer(settings.kafka)
print(f"    kafka producer -> {type(kafka_producer).__name__} (PLAINTEXT defaults)")

# Redpanda defaults to SASL_SSL + SCRAM-SHA-256 (Redpanda Cloud), so client
# creation requires credentials. They normally come from
# FRAUDSHIELD_KAFKA_SASL_USERNAME / _PASSWORD; here we pass them as overrides.
# The extra overrides keep this offline demo quiet: they stop the client from
# retry-looping against the local PLAINTEXT broker in the background.
redpanda_producer = get_broker_factory("redpanda").create_producer(
    settings.kafka,
    **{
        "sasl.username": "demo-user",
        "sasl.password": "demo-secret",
        "log_level": 2,
        "socket.connection.setup.timeout.ms": 1000,
        "reconnect.backoff.ms": 30000,
        "reconnect.backoff.max.ms": 60000,
    },
)
print(f" redpanda producer -> {type(redpanda_producer).__name__} (SASL_SSL + SCRAM-SHA-256)")

Configured broker type: kafka
    kafka producer -> Producer (PLAINTEXT defaults)
 redpanda producer -> Producer (SASL_SSL + SCRAM-SHA-256)


%3|1790888607.004|FAIL|fraudshield-transaction-producer#producer-1| [thrd:localhost:9092/bootstrap]: localhost:9092/bootstrap: Connect to ipv6#[::1]:9092 failed: Connection refused (after 5ms in state CONNECT)


## 2. Stateful Streaming Features

`StatefulFeatureStore` computes rolling counts/sums/means per user, merchant,
currency, and status in O(1) amortized time per event. Watch the 1h/24h
counters react when the simulated user suddenly starts moving large amounts.

In [3]:
from fraudshield.feature_engineering.stateful_aggregates import StatefulFeatureStore

store = StatefulFeatureStore()
start = pd.Timestamp("2026-08-01 09:00:00", tz="UTC")

rows = []
for index in range(60):
    payload = {
        "user_id": "demo_user",
        "merchant_id": f"merchant_{index % 7}",
        "amount": 250.0 if index < 55 else 4800.0,
        "currency": "USD",
        "status": "approved",
        "transaction_date": start + pd.Timedelta(minutes=7 * index),
    }
    rows.append(store.build_features(payload))

feature_frame = pd.DataFrame(rows)
feature_frame.tail(6)

,user_time_since_last_txn,user_amount_zscore,user_txn_count_1h,user_amount_sum_1h,user_amount_mean_1h,merchant_txn_count_1h,merchant_amount_mean_1h,currency_txn_count_1h,status_txn_count_1h,user_txn_count_24h,...,merchant_amount_mean_7d,currency_txn_count_7d,status_txn_count_7d,user_txn_count_30d,user_amount_sum_30d,user_amount_mean_30d,merchant_txn_count_30d,merchant_amount_mean_30d,currency_txn_count_30d,status_txn_count_30d
54,420.0,0.000000,8.0,2000.0,250.00,1.0,250.0,8.0,8.0,54.0,...,250.0,54.0,54.0,54.0,13500.0,250.000000,7.0,250.0,54.0,54.0
55,420.0,0.000000,8.0,2000.0,250.00,1.0,250.0,8.0,8.0,55.0,...,250.0,55.0,55.0,55.0,13750.0,250.000000,7.0,250.0,55.0,55.0
56,420.0,7.349684,8.0,6550.0,818.75,1.0,250.0,8.0,8.0,56.0,...,250.0,56.0,56.0,56.0,18550.0,331.250000,8.0,250.0,56.0,56.0
57,420.0,5.197840,8.0,11100.0,1387.50,1.0,250.0,8.0,8.0,57.0,...,250.0,57.0,57.0,57.0,23350.0,409.649123,8.0,250.0,57.0,57.0
58,420.0,4.244672,8.0,15650.0,1956.25,1.0,250.0,8.0,8.0,58.0,...,250.0,58.0,58.0,58.0,28150.0,485.344828,8.0,250.0,58.0,58.0
59,420.0,3.676540,8.0,20200.0,2525.00,1.0,250.0,8.0,8.0,59.0,...,250.0,59.0,59.0,59.0,32950.0,558.474576,8.0,250.0,59.0,59.0


## 3. Batch Rolling Features (Same Engine As Preprocessing)

`add_transaction_features` builds the same window features over a DataFrame,
with leakage prevention (`closed="left"`, `shift(1)` z-scores).

In [4]:
from fraudshield.feature_engineering.transaction_features import (
    TransactionFeatureConfig,
    add_transaction_features,
)

sample_df = pd.read_csv(
    Path("data/raw/synthetic_fraud_data.csv"), parse_dates=["transaction_date"]
).head(2000)
featured_df = add_transaction_features(sample_df, TransactionFeatureConfig())

new_columns = [c for c in featured_df.columns if c not in sample_df.columns]
print("Added columns:", new_columns)
featured_df[new_columns[:6]].head()

Added columns: ['user_time_since_last_txn', 'user_amount_zscore', 'user_txn_count_1h', 'user_amount_sum_1h', 'user_amount_mean_1h', 'user_txn_count_24h', 'user_amount_sum_24h', 'user_amount_mean_24h', 'user_txn_count_7d', 'user_amount_sum_7d', 'user_amount_mean_7d', 'user_txn_count_30d', 'user_amount_sum_30d', 'user_amount_mean_30d', 'merchant_txn_count_1h', 'merchant_amount_mean_1h', 'merchant_txn_count_24h', 'merchant_amount_mean_24h', 'merchant_txn_count_7d', 'merchant_amount_mean_7d', 'merchant_txn_count_30d', 'merchant_amount_mean_30d', 'currency_txn_count_1h', 'currency_txn_count_24h', 'currency_txn_count_7d', 'currency_txn_count_30d', 'status_txn_count_1h', 'status_txn_count_24h', 'status_txn_count_7d', 'status_txn_count_30d']


,user_time_since_last_txn,user_amount_zscore,user_txn_count_1h,user_amount_sum_1h,user_amount_mean_1h,user_txn_count_24h
0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0


## 4. Fraud Ring Detection

Rings are communities found with the Louvain algorithm over the 2-hop
neighborhood of an account in Neo4j (accounts linked through shared devices
and IPs). Below we build a synthetic subgraph in-memory; communities smaller
than `min_ring_size` are dropped.

In [5]:
from fraudshield.graph.fraud_ring_detector import FraudRingDetector

records = [
    {"neighbor_id": "acct_2", "shared_id": "device_alpha", "edge_weight": 2},
    {"neighbor_id": "acct_3", "shared_id": "device_alpha", "edge_weight": 1},
    {"neighbor_id": "acct_4", "shared_id": "ip_beta", "edge_weight": 1},
    {"neighbor_id": "acct_5", "shared_id": "ip_beta", "edge_weight": 3},
    {"neighbor_id": "acct_6", "shared_id": "lonely_device", "edge_weight": 1},
]

graph = FraudRingDetector._build_graph("acct_1", records)
detector = FraudRingDetector(driver=None, min_ring_size=3)
rings = detector._find_communities(graph)

for ring in rings:
    print(ring)

FraudRing(ring_id='RING_352f3f1d4bca', member_accounts=['acct_1', 'acct_4', 'acct_5', 'acct_6'], shared_entities=['device_alpha', 'ip_beta', 'lonely_device'], risk_score=0.5507, density=0.6667)


In [6]:
from fraudshield.graph.graph_builder.builder import FraudGraphBuilder

builder = FraudGraphBuilder()
if builder.ring_detector is not None:
    live_rings = builder.ring_detector.detect_rings("acct_1")
    print("Live rings:", live_rings)
else:
    print("Neo4j is offline - live detection skipped. Start the stack with:")
    print("  docker compose -f infra/docker-compose.yml up -d")
builder.close()

Failed to initialize Neo4j Graph Builder connection natively: {neo4j_code: Neo.ClientError.Security.Unauthorized} {message: Unsupported authentication token, missing key `credentials`} {gql_status: 50N42} {gql_status_description: error: general processing exception - unexpected error. Unsupported authentication token, missing key `credentials`}


Neo4j is offline - live detection skipped. Start the stack with:
  docker compose -f infra/docker-compose.yml up -d


## 5. Hybrid Risk Engine

`HybridRiskEngine` blends the ML probability (0.6), the graph/ring score
(0.3), and normalized rule breaches (0.1). When a ring detector is attached,
its account score is folded into the graph component.

In [7]:
from unittest.mock import MagicMock

from fraudshield.core.risk_engine.engine import (
    HybridRiskEngine,
    count_rule_breaches,
    extract_account_id,
)

ring_stub = MagicMock()
ring_stub.assess_account.return_value = 0.85

engine = HybridRiskEngine(ring_detector=ring_stub)

risky_payload = {
    "user_id": "acct_900",
    "amount": 9500.0,
    "is_international": True,
    "is_online": True,
}
risky = engine.evaluate_transaction(
    ml_score=0.78,
    graph_score=0.66,
    rules_breached=count_rule_breaches(risky_payload),
    account_id=extract_account_id(risky_payload),
)
benign = engine.evaluate_transaction(ml_score=0.05, graph_score=0.10, rules_breached=0)

pd.DataFrame([risky, benign], index=["risky", "benign"]).T

,risky,benign
composite_fraud_score,0.7897,0.06
ml_contribution,0.78,0.05
graph_contribution,0.85,0.1
rule_contribution,0.6667,0.0
assigned_risk_level,HIGH,LOW
action,BLOCK,ALLOW


## 6. Inference API + Prometheus Metrics

The FastAPI app serves `/predict`, `/health`, and the Prometheus `/metrics`
endpoint. This runs the same hybrid assessment as the streaming consumer.

In [8]:
from fastapi.testclient import TestClient

from fraudshield.ml.inference.api import create_app

app = create_app()
with TestClient(app) as client:
    predict_response = client.post(
        "/predict",
        json={
            "transaction_id": "NB_STREAM_0001",
            "account_id": "acct_42",
            "amount": 3800.0,
            "is_international": True,
            "is_online": True,
        },
    )
    health_response = client.get("/health")
    metrics_response = client.get("/metrics/")

print("predict:", predict_response.json())
print("health :", health_response.json())

fraudshield_lines = [
    line for line in metrics_response.text.splitlines() if line.startswith("fraudshield_")
]
print()
print("\n".join(fraudshield_lines[:12]))

%3|1790888608.004|FAIL|fraudshield-transaction-producer#producer-1| [thrd:localhost:9092/1]: localhost:9092/1: Connect to ipv6#[::1]:9092 failed: Connection refused (after 0ms in state CONNECT)


/Users/mudit/Developer/FraudShield/.venv/lib/python3.13/site-packages/fastapi/testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


/Users/mudit/Developer/FraudShield/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-10-02 02:33:30,359 INFO [fraudshield.ml.inference.service] Loaded inference artifacts for model xgboost


2026-10-02 02:33:30,361 INFO [fraudshield.ml.explainability.shap_explainer] XGBoost model detected, using native pred_contribs for explanations.


2026-10-02 02:33:30,380 WARNING [fraudshield.graph.graph_builder.builder] Failed to initialize Neo4j Graph Builder connection natively: {neo4j_code: Neo.ClientError.Security.Unauthorized} {message: Unsupported authentication token, missing key `credentials`} {gql_status: 50N42} {gql_status_description: error: general processing exception - unexpected error. Unsupported authentication token, missing key `credentials`}


2026-10-02 02:33:30,414 INFO [httpx] HTTP Request: POST http://testserver/predict "HTTP/1.1 200 OK"


2026-10-02 02:33:30,416 INFO [httpx] HTTP Request: GET http://testserver/health "HTTP/1.1 200 OK"


2026-10-02 02:33:30,417 INFO [httpx] HTTP Request: GET http://testserver/metrics/ "HTTP/1.1 200 OK"


predict: {'transaction_id': 'NB_STREAM_0001', 'fraud_probability': 0.12216313183307648, 'risk_level': 'LOW', 'action': 'ALLOW', 'model_loaded': True, 'source': 'trained_model', 'explanation': {'num__is_international': 1.4289, 'num__currency_txn_count_24h': 1.1239, 'num__amount': 1.0638, 'num__user_amount_sum_7d': -1.0174, 'num__status_txn_count_30d': 1.0067}}
health : {'status': 'healthy', 'model_loaded': True, 'model_name': 'xgboost'}

fraudshield_transactions_total{source="trained_model",status="processed"} 1.0
fraudshield_transactions_created{source="trained_model",status="processed"} 1.790888610407347e+09
fraudshield_risk_level_total{level="LOW"} 1.0
fraudshield_risk_level_created{level="LOW"} 1.790888610407337e+09
fraudshield_inference_latency_seconds_bucket{le="0.001",model_name="xgboost",source="trained_model"} 0.0
fraudshield_inference_latency_seconds_bucket{le="0.005",model_name="xgboost",source="trained_model"} 0.0
fraudshield_inference_latency_seconds_bucket{le="0.01",model_

## 7. Running The Full Stack

```bash
cp .env.example infra/.env   # then replace every placeholder with a unique secret
docker compose -f infra/docker-compose.yml up -d --build
```

| Service | Port | Purpose |
|---|---|---|
| fraudshield | 8000 | Inference API + `/metrics` |
| kafka | 9092 | Transaction topic |
| neo4j | 7474 / 7687 | Entity graph + fraud rings |
| postgres | 5432 | Batch warehouse |
| prometheus | 9091 | Metrics collection |
| grafana | 3000 | Dashboards |

Stream synthetic traffic once the stack is healthy:

```bash
python -m fraudshield.streaming.transaction_producer   # publish events
python -m fraudshield.streaming.kafka_consumer         # score + graph + metrics
```